<a href="https://colab.research.google.com/github/shinnew9/CSE498_AI-Healthcare-Robotics/blob/main/Lab1_ObjectDetection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# setting up a GPU

!nvidia-smi

Sun Sep  6 05:44:36 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   55C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# I mounted the google drive to access to my drive

from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pwd

/content


In [ ]:
%cd /content/drive/MyDrive/ProfChuah/CSE498-AIHealthcare_Robotics

/content/drive/MyDrive/ProfChuah/CSE498-AIHealthcare_Robotics


### step 1

I will git clone in this space

In [ ]:
# !git clone https://github.com/valeriouberti/webcam-object-recognition.git

In [ ]:
# %pip install "ultralytics<=8.3.40" supervision roboflow

In [ ]:
import os
from ultralytics import YOLO

# Base directory for this assignment
base_dir = "/content/drive/MyDrive/ProfChuah/CSE498-AIHealthcare_Robotics"

# Subdirectories built from base_dir
webcam_images_dir = os.path.join(base_dir, "webcam_images")
results_dir = os.path.join(base_dir, "results")

HOME = base_dir
os.chdir(HOME)
print("Current directory:", os.getcwd())

ModuleNotFoundError: No module named 'ultralytics'

In [ ]:
def get_sample_images(folder, num_images=5):
    """Return the first num_images image file paths from a folder."""
    valid_ext = (".jpg", ".jpeg", ".png")
    all_images = sorted(
        os.path.join(folder, f)
        for f in os.listdir(folder)
        if f.lower().endswith(valid_ext)
    )
    return all_images[:num_images]


def run_inference(model_path, image_paths, save_name):
    """Run YOLO inference on given images and save annotated results under results_dir/save_name."""
    model = YOLO(model_path)
    results = model.predict(image_paths, conf=0.25, save=True, project=results_dir, name=save_name)
    for r in results:
        r.show()
    return results

In [ ]:
# Pick 5 sample images from my 50 webcam images
test_images = get_sample_images(webcam_images_dir, num_images=5)
print("Selected test images:", test_images)

# Step 1: baseline test with the pretrained (not fine-tuned) model
baseline_results = run_inference("yolo11n.pt", test_images, save_name="baseline")

In [ ]:
# from ultralytics import YOLO

# model = YOLO("webcam-object-recognition/models/yolo11n.pt")

# # trying on few items
# result1 = model.predict("./webcam_images/oliveoil.jpg", conf=0.5)
# result1[0].show()

In [ ]:
# result2 = model.predict("./webcam_images/pencilholder.jpg", conf=0.5)
# result2[0].show()

In [ ]:
# result3 = model.predict("./webcam_images/kitchentowel.jpg", conf=0.5)
# result3[0].show()

### Step 2 - Objection detection on custom dataset
following this [link](https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/train-yolo11-object-detection-on-custom-dataset.ipynb
)

In [ ]:
# 1st, I will download the dataset from kaggle

!mkdir -p data
import kagglehub
path = kagglehub.dataset_download("elvinrustam/grocery-dataset")
print(path)

In [ ]:
import shutil, os

dest_path = "//content/drive/MyDrive/ProfChuah/CSE498-AIHealthcare_Robotics/grocery-dataset"

# using the same paths
shutil.copytree(path, dest_path, dirs_exist_ok=True)

print("Copied files:", os.listdir(dest_path))
print(os.listdir(dest_path))

In [ ]:
import pandas as pd

df = pd.read_csv(f"{dest_path}/GroceryDataset.csv")
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
df.head()

In [ ]:
# Following the instruction on the link from cell after working on RoboFlow

from google.colab import userdata
from roboflow import Roboflow

ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
rf = Roboflow(api_key=ROBOFLOW_API_KEY)

workspace = rf.workspace("yoojin-shin")
project = workspace.project("lab1-uopop")
version = project.version(1)
dataset = version.download("yolov11")

In [ ]:
%cd {HOME}

!yolo task=detect mode=train model=yolo11s.pt data={dataset.location}/data.yaml epochs=10 imgsz=640 plots=True

In [ ]:
import os

save_dir = os.path.join(HOME, "runs", "detect", "train2")
weights_path = os.path.join(save_dir, "weights", "best.pt")
print(weights_path)

print(os.path.exists(weights_path))  # True가 나와야 함

In [ ]:
finetuned_results = run_inference(weights_path, test_images, save_name="after_finetune")

In [ ]:
from ultralytics import YOLO

model = YOLO(weights_path)  # load the fine-tuned model (Using the same weights_path, that I have made in cell 25)

# Lower the confidence threshold to see if there are weak (low-confidence) detections
low_conf_results = model.predict(test_images, conf=0.01)
for r in low_conf_results:
    print(r.boxes.cls, r.boxes.conf)

## Notes: Step 1 & 2 (issues & open questions)

- Kaggle Grocery Dataset had no images/annotations (text-only) → wasn't able to use it; created a custom dataset instead by manually labeling my own 50 photos in Roboflow.
- Dataset has 51 classes across only 50 images (~1 image/class) → very little data per class for fine-tuning.
- Fine-tuned model (10 epochs) scored 0 on several classes (marker, slippers, sunglasses, tumbler) in validation — likely too few examples to learn from.
- At conf=0.25, the fine-tuned model detected **nothing** on the 5 held-out test images (same ones used in Step 1 baseline). At conf=0.01, weak detections appeared (top confidence ~0.22–0.88 per image), so it's not pure random noise, but not confident enough to be usable.
- **Open question**: is the failure mainly due to (a) too few images per class, (b) too few epochs, or (c) the test images not matching classes seen in training? Not fully isolated yet.

### Step 3 - Zero shot
following this [link](https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/zero-shot-object-detection-and-segmentation-with-yoloe.ipynb)

In [ ]:
# Install YOLOE and its dependencies
!pip install -q "git+https://github.com/THU-MIG/yoloe.git#subdirectory=third_party/CLIP"
!pip install -q "git+https://github.com/THU-MIG/yoloe.git#subdirectory=third_party/ml-mobileclip"
!pip install -q "git+https://github.com/THU-MIG/yoloe.git#subdirectory=third_party/lvis-api"
!pip install -q "git+https://github.com/THU-MIG/yoloe.git"
!pip install -q supervision

# Download the CLIP-based text encoder weights (needed for text prompts)
!wget -q https://docs-assets.developer.apple.com/ml-research/datasets/mobileclip/mobileclip_blt.pt -P {HOME}

In [ ]:
from huggingface_hub import hf_hub_download

# Download the YOLOE model weights (pretrained, no fine-tuning needed for zero-shot)
yoloe_weights = hf_hub_download(repo_id="jameslahm/yoloe", filename="yoloe-v8l-seg.pt", local_dir=HOME)
print(yoloe_weights)